# Event Stream Validation

Lightweight inspection of a landed Bronze and Silver sample from a bounded pipeline run.

**This notebook is a validation surface, not a pipeline entrypoint.**
It reads already-landed data from `data/bronze/raw/` and `data/silver/tables/`.
No pipeline logic runs here.

Before running the cells below, complete at least steps 3–5 of the local run:

```
python src/jobs/run_publisher.py --max-events 100 --max-seconds 60
python src/jobs/run_bronze_consumer.py --max-events 100 --max-seconds 60
python src/jobs/run_silver.py
```

See `docs/local-run.md` for the full run sequence.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

def resolve_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidate in candidates:
        if (candidate / "src").exists() and (candidate / "data").exists():
            return candidate
        if (candidate / "projects/05-event-stream-analytics/src").exists():
            return candidate / "projects/05-event-stream-analytics"
    raise RuntimeError("Could not resolve the project root for project 05.")

PROJECT_ROOT = resolve_project_root()
PROJECT_SRC = PROJECT_ROOT / "src"
if str(PROJECT_SRC) not in sys.path:
    sys.path.insert(0, str(PROJECT_SRC))

from config import get_settings
from stream.messages import load_jsonl

settings = get_settings()
print(PROJECT_ROOT)


/home/willian/PhaifferTech/repos/data-engineering-portfolio/projects/05-event-stream-analytics


In [2]:
# Data availability check — run this before the inspection cells below.
# If Bronze or Silver data is missing, the cells that follow will print empty results.

_bronze_raw_dir = settings.bronze_raw_dir
_silver_tables_dir = settings.silver_tables_dir / "recentchange_events"

_has_bronze = _bronze_raw_dir.exists() and any(_bronze_raw_dir.rglob("batch_*.jsonl"))
_has_silver = _silver_tables_dir.exists() and any(_silver_tables_dir.rglob("*.parquet"))

if not _has_bronze and not _has_silver:
    print("No landed data found. Run the full pipeline first:")
    print()
    print("  python src/jobs/run_publisher.py --max-events 100 --max-seconds 60")
    print("  python src/jobs/run_bronze_consumer.py --max-events 100 --max-seconds 60")
    print("  python src/jobs/run_silver.py")
    print("  python src/jobs/run_gold.py")
elif not _has_bronze:
    print("No Bronze batches found. Run the publisher and Bronze consumer first:")
    print()
    print("  python src/jobs/run_publisher.py --max-events 100 --max-seconds 60")
    print("  python src/jobs/run_bronze_consumer.py --max-events 100 --max-seconds 60")
elif not _has_silver:
    print("Bronze data found but no Silver files yet. Run the Silver job:")
    print()
    print("  python src/jobs/run_silver.py")
else:
    print("Bronze and Silver data found. Ready to inspect.")


Bronze and Silver data found. Ready to inspect.


In [3]:
bronze_batches = sorted(settings.bronze_raw_dir.rglob("batch_*.jsonl"))
silver_partitions = sorted((settings.silver_tables_dir / "recentchange_events").rglob("*.parquet"))

print(f"Bronze batches: {len(bronze_batches)}")
print(f"Silver files: {len(silver_partitions)}")
bronze_batches[-3:], silver_partitions[-3:]


Bronze batches: 1
Silver files: 1


([PosixPath('/home/willian/PhaifferTech/repos/data-engineering-portfolio/projects/05-event-stream-analytics/data/bronze/raw/stream_date=2026-04-15/batch_20260415T185237Z.jsonl')],
 [PosixPath('/home/willian/PhaifferTech/repos/data-engineering-portfolio/projects/05-event-stream-analytics/data/silver/tables/recentchange_events/event_date=2026-04-15/batch_20260415T185237Z.parquet')])

In [4]:
if bronze_batches:
    sample_records = load_jsonl(bronze_batches[-1])
    bronze_df = pd.json_normalize(sample_records)
    display(bronze_df[[
        "event.meta.dt",
        "event.type",
        "event.wiki",
        "event.namespace",
        "event.user",
        "broker.partition",
        "broker.offset",
    ]].head(10))
else:
    print("No Bronze batch found yet. Run the publisher and Bronze consumer first.")


,event.meta.dt,event.type,event.wiki,event.namespace,event.user,broker.partition,broker.offset
0,2026-04-15T18:52:31.592Z,categorize,commonswiki,14,DPLA bot,0,0
1,2026-04-15T18:52:31.618Z,categorize,commonswiki,14,DPLA bot,0,1
2,2026-04-15T18:52:31.598Z,edit,wikidatawiki,0,PARAKANYAA,0,2
3,2026-04-15T18:52:31.613Z,edit,enwiktionary,0,SurjectionBot,0,3
4,2026-04-15T18:52:31.644Z,categorize,ruwikinews,14,AtUkrBot,0,4


In [5]:
if silver_partitions:
    silver_df = pd.read_parquet(silver_partitions[-1])
    display(silver_df[[
        "event_timestamp",
        "event_type",
        "actor_segment",
        "wiki",
        "namespace",
        "title",
    ]].head(10))
    display(silver_df["event_type"].value_counts(dropna=False).head(10))
else:
    print("No Silver files found yet. Run the Silver job after Bronze landing.")


,event_timestamp,event_type,actor_segment,wiki,namespace,title
0,2026-04-15 18:52:31.592,categorize,bot,commonswiki,14,Category:Information field template with forma...
1,2026-04-15 18:52:31.618,categorize,bot,commonswiki,14,Category:Media contributed by the Digital Publ...
2,2026-04-15 18:52:31.598,edit,human,wikidatawiki,0,Q136538413
3,2026-04-15 18:52:31.613,edit,bot,enwiktionary,0,askeettinen
4,2026-04-15 18:52:31.644,categorize,human,ruwikinews,14,Категория:Спорт на Украине


event_type
categorize    3
edit          2
Name: count, dtype: int64